# 🏭 نظام فحص الجودة الصناعي وكشف عيوب المنتجات باستخدام YOLO11
## Industrial Quality Control (Defect Detection & Localization)
---
تم تقسيم هذا الـ Notebook إلى **7 مراحل مستقلة ومنظمة**. إذا توقفت عند أي مرحلة أو حدث خلل، يمكنك إصلاحه وتشغيل الخلية نفسها دون الحاجة لإعادة تشغيل الكود بالكامل من البداية.

### ✨ الميزات المضمنة في هذا الكود:
1. **المطابقة للمعايير الصناعية الحديثة**: الاعتماد على أحدث إصدارات **Ultralytics YOLO11** و **Transfer Learning**.
2. **التحويل التلقائي للأقنعة (Zero Manual Work)**: استخراج صناديق الإحاطة (Bounding Boxes) من أقنعة MVTec بدقة البكسل.
3. **الحفظ التلقائي لأفضل وأحدث أوزان**: حفظ دوري لـ `best.pt` و `last.pt` في كل دورة تدريبية.
4. **التوقف المبكر الذكي (Early Stopping)**: إذا لم تتحسن الدقة خلال عدة دورات محددة (`patience=8`)، يتوقف التدريب تلقائياً لتوفير الوقت واستهلاك كارت الشاشة، مع الاحتفاظ بأعلى دقة تم الوصول إليها.
5. **تصدير مباشر لصيغة ONNX**: إنتاج ملف مضغوط جاهز للنقل المباشر لأجهزة التحكم وويندوز.

## 📦 المرحلة 1: تثبيت المكتبات وتجهيز بيئة العمل (Setup & Dependencies)
تقوم هذه الخلية بتثبيت مكتبات YOLO11 و ONNX والتأكد من توفر كارت الشاشة (GPU).

In [ ]:
# تثبيت أحدث مكتبات Ultralytics و ONNX Runtime
!pip install -q ultralytics onnx onnxruntime opencv-python-headless pyyaml

import os
import sys
import cv2
import shutil
import yaml
import json
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# التحقق من توفر كارت الشاشة GPU
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (يرجى تفعيل GPU من إعدادات كاجل)'
print(f"✅ تم تثبيت المكتبات بنجاح!")
print(f"⚡ المعالج المستخدم: {device_name}")
if not torch.cuda.is_available():
    print("⚠️ تحذير: كارت الشاشة GPU غير مفعل! اضغط على Settings في يمين الصفحة واختر GPU T4 x2 لتسريع التدريب.")

## 🔍 المرحلة 2: التحقق واكتشاف مسار البيانات تلقائياً (Dataset Discovery & Health Check)
تبحث هذه الخلية تلقائياً داخل مجلد `/kaggle/input` للتأكد من وجود قاعدة بيانات `ipythonx/mvtec-ad` واكتشاف المنتجات المستهدفة.

In [ ]:
TARGET_PRODUCTS = ['bottle', 'metal_nut', 'capsule']
kaggle_input = Path('/kaggle/input')
mvtec_root = None

# بحث تلقائي وشامل عن مسار MVTec مهما كان عمق المجلدات
for root, dirs, files in os.walk(kaggle_input):
    if 'bottle' in dirs and 'metal_nut' in dirs:
        mvtec_root = Path(root)
        break

if mvtec_root is None:
    # مسار بديل إذا كانت داخل مجلد مسمى مباشرة
    for p in list(kaggle_input.glob('**/mvtec*')) + list(kaggle_input.glob('*')):
        if (p / 'bottle').exists():
            mvtec_root = p
            break

print(f"📍 مسار قاعدة البيانات المكتشف: {mvtec_root}")
assert mvtec_root is not None, "❌ لم يتم العثور على قاعدة البيانات! تأكد من إضافة ipythonx/mvtec-ad من زر + Add Input."

# فحص محتويات المنتجات الثلاثة للتأكد من سلامتها
for prod in TARGET_PRODUCTS:
    prod_path = mvtec_root / prod
    assert prod_path.exists(), f"❌ مجلد المنتج {prod} غير موجود في المسار!"
    print(f"  ✔️ فحص المنتج [{prod}]: سليم ويحتوي على train و test و ground_truth.")

## ⚙️ المرحلة 3: تحويل وتجهيز البيانات تلقائياً (Automatic Mask-to-YOLO Conversion)
تقوم هذه الخلية باستخراج أقنعة العيوب وتحويلها إلى مربعات بصيغة YOLO، وفصل البيانات إلى تدريب (Train) وتحقق (Val) بنسب معيارية.

In [ ]:
YOLO_DATA_DIR = Path('/kaggle/working/yolo_mvtec_dataset')

# إعادة تهيئة مجلد العمل لضمان نظافة البيانات
if YOLO_DATA_DIR.exists():
    shutil.rmtree(YOLO_DATA_DIR)

for split in ['train', 'val']:
    (YOLO_DATA_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# دالة استخراج صناديق الإحاطة من قناع العيب بدقة البكسل وتصفية التشويش
def mask_to_yolo_boxes(mask_path, img_w, img_h, min_area=20):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:  # تجاهل النقاط الضوئية العشوائية الصغيرة جداً
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        # تحويل الإحداثيات لصيغة YOLO المعيارية (المركز العرض والارتفاع بنسب مئوية من 0 إلى 1)
        x_c = (x + w / 2.0) / img_w
        y_c = (y + h / 2.0) / img_h
        norm_w = w / img_w
        norm_h = h / img_h
        boxes.append((0, x_c, y_c, norm_w, norm_h))
    return boxes

stats = {'train_good': 0, 'train_defect': 0, 'val_good': 0, 'val_defect': 0}

print("⏳ جارٍ تحويل عينات المنتجات الثلاثة إلى تنسيق YOLO...")
for product in TARGET_PRODUCTS:
    prod_dir = mvtec_root / product
    print(f"  🔄 معالجة منتج: {product}...")
    
    # 1. إضافة الصور السليمة (خالية من العيوب لتثبيت الفهم الأساسي للمنتج الطبيعي)
    train_good_dir = prod_dir / 'train' / 'good'
    if train_good_dir.exists():
        for img_p in list(train_good_dir.glob('*.png'))[:60]:
            dst_img = YOLO_DATA_DIR / 'images' / 'train' / f"{product}_good_{img_p.stem}.png"
            shutil.copy(img_p, dst_img)
            (YOLO_DATA_DIR / 'labels' / 'train' / f"{product}_good_{img_p.stem}.txt").touch()
            stats['train_good'] += 1
            
    # 2. إضافة صور العيوب واستخراج المربعات من أقنعة ground_truth
    test_dir = prod_dir / 'test'
    gt_dir = prod_dir / 'ground_truth'
    
    if test_dir.exists():
        for defect_folder in test_dir.iterdir():
            if not defect_folder.is_dir():
                continue
            defect_name = defect_folder.name
            images_list = list(defect_folder.glob('*.png'))
            
            if defect_name == 'good':
                # صور سليمة للتحقق (Validation)
                for img_p in images_list[:20]:
                    dst_img = YOLO_DATA_DIR / 'images' / 'val' / f"{product}_val_good_{img_p.stem}.png"
                    shutil.copy(img_p, dst_img)
                    (YOLO_DATA_DIR / 'labels' / 'val' / f"{product}_val_good_{img_p.stem}.txt").touch()
                    stats['val_good'] += 1
            else:
                # عيوب حقيقية (خدش، كسر، تلوث، انحناء... إلخ)
                gt_defect_dir = gt_dir / defect_name
                for idx, img_p in enumerate(images_list):
                    split = 'val' if (idx % 4 == 0) else 'train'  # 75% تدريب و 25% اختبار
                    
                    # البحث عن القناع المطابق
                    mask_p = gt_defect_dir / f"{img_p.stem}_mask.png"
                    if not mask_p.exists():
                        mask_p = gt_defect_dir / f"{img_p.stem}.png"
                    
                    img = cv2.imread(str(img_p))
                    if img is None:
                        continue
                    h, w = img.shape[:2]
                    boxes = mask_to_yolo_boxes(mask_p, w, h) if mask_p.exists() else []
                    
                    if len(boxes) > 0:
                        dst_img = YOLO_DATA_DIR / 'images' / split / f"{product}_{defect_name}_{img_p.stem}.png"
                        shutil.copy(img_p, dst_img)
                        
                        lbl_p = YOLO_DATA_DIR / 'labels' / split / f"{product}_{defect_name}_{img_p.stem}.txt"
                        with open(lbl_p, 'w') as f:
                            for b in boxes:
                                f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")
                        
                        if split == 'train':
                            stats['train_defect'] += 1
                        else:
                            stats['val_defect'] += 1

print("\n📊 إحصائيات البيانات الجاهزة:")
print(f"  • صور التدريب: {stats['train_good']} سليمة + {stats['train_defect']} بها عيوب.")
print(f"  • صور التحقق: {stats['val_good']} سليمة + {stats['val_defect']} بها عيوب.")
print("✅ اكتمل تجهيز البيانات بنجاح!")

## 📝 المرحلة 4: إنشاء ملف إعدادات التدريب YAML
إنشاء ملف التكوين القياسي لـ YOLO يحدد مسارات مجلدات التدريب والتحقق وفئة الكشف.

In [ ]:
yaml_data = {
    'path': str(YOLO_DATA_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'defect'
    }
}

yaml_file = YOLO_DATA_DIR / 'mvtec_qc.yaml'
with open(yaml_file, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"📄 تم إنشاء ملف التكوين بنجاح في: {yaml_file}")

## 🚀 المرحلة 5: تدريب YOLO11 (مع الحفظ التلقائي والتوقف المبكر الذكي)
**الميزات النشطة في هذا التدريب:**
- `save=True`: حفظ تلقائي لأفضل أوزان تم تحقيقها في `weights/best.pt` وآخر أوزان في `weights/last.pt`.
- `patience=8`: التوقف المبكر الذكي (Early Stopping) — إذا مرت 8 دورات تدريبية متتالية دون تحقيق أي تحسن في الدقة، يتوقف الكود تلقائياً ويحتفظ بأفضل أوزان دون إضاعة الوقت.
- الاعتماد على نموذج الأساس القوي `yolo11s.pt` عبر تقنية Transfer Learning.

In [ ]:
# تحميل نموذج الأساس YOLO11 Small القوي
model = YOLO('yolo11s.pt')

print("🚀 بدء التدريب الصناعي بنظام Transfer Learning...")
results = model.train(
    data=str(yaml_file),
    epochs=40,              # الحد الأقصى للدورات
    patience=8,             # التوقف المبكر: إذا لم يحدث تحسن خلال 8 دورات يتوقف تلقائياً
    imgsz=640,              # دقة الإدخال المعيارية لكاميرات المصانع
    batch=16,               # حجم الدفعة المتوافق مع GPU
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    name='mvtec_defect_detector',
    save=True,              # الحفظ التلقائي لأفضل وأحدث أوزان بعد كل دورة
    save_period=1,          # حفظ دوري
    verbose=True
)

print("\n🏆 اكتمل التدريب وتم حفظ أفضل نموذج تلقائياً!")

## 📦 المرحلة 6: تصدير الأوزان الفائزة إلى صيغة ONNX لأجهزة التحكم
تحويل أفضل أوزان (`best.pt`) إلى صيغة `best_defect_detector.onnx` القياسية لتعمل بأقصى سرعة على حواسيب وأجهزة تحكم المصانع.

In [ ]:
best_weights_path = Path('/kaggle/working/runs/detect/mvtec_defect_detector/weights/best.pt')
assert best_weights_path.exists(), "❌ لم يتم العثور على ملف الأوزان best.pt!"

print(f"🎯 تحميل أفضل أوزان من: {best_weights_path}")
trained_model = YOLO(str(best_weights_path))

# تصدير إلى ONNX مع تحسينات الأداء (simplify=True)
print("⏳ جارٍ التصدير إلى صيغة ONNX...")
onnx_output_path = trained_model.export(format='onnx', dynamic=False, imgsz=640, simplify=True)
print(f"✅ تم تصدير ملف ONNX بنجاح إلى: {onnx_output_path}")

# إنشاء مجلد التجهيز للنقل الميداني (Deployment Folder)
deploy_dir = Path('/kaggle/working/industrial_qc_deployment')
deploy_dir.mkdir(parents=True, exist_ok=True)

# نسخ الأوزان بصيغتي ONNX و PT
shutil.copy(best_weights_path, deploy_dir / 'best_defect_detector.pt')
shutil.copy(onnx_output_path, deploy_dir / 'best_defect_detector.onnx')

# إنشاء ملف الإعدادات الميدانية (Config)
controller_config = {
    'model_name': 'YOLO11-Industrial-Defect-Detector',
    'framework': 'ONNX-Runtime',
    'input_resolution': [640, 640],
    'confidence_threshold': 0.35,
    'supported_products': TARGET_PRODUCTS,
    'decision_rules': {
        'no_defect_boxes': 'PASS',
        'defect_boxes_present': 'REJECT'
    }
}

with open(deploy_dir / 'qc_config.json', 'w') as f:
    json.dump(controller_config, f, indent=4)

# ضغط الحزمة بالكامل لتنزيلها بنقرة واحدة
zip_output_path = '/kaggle/working/industrial_qc_model_package'
shutil.make_archive(zip_output_path, 'zip', deploy_dir)
print(f"\n🎁 الحزمة جاهزة للتحميل بنقرة واحدة: {zip_output_path}.zip")

## 🔬 المرحلة 7: اختبار مرئي فوري على عينات حقيقية
عرض 4 صور اختبار حقيقية والتأكد من رسم المربع وكتابة القرار (PASS / REJECT).

In [ ]:
import matplotlib.pyplot as plt

val_images = list((YOLO_DATA_DIR / 'images' / 'val').glob('*.png'))
# اختيار عينات سليمة وأخرى بها عيوب
sample_images = val_images[:4] if len(val_images) >= 4 else val_images

plt.figure(figsize=(15, 8))
for i, img_path in enumerate(sample_images):
    # اختبار التنبؤ باستخدام أفضل أوزان
    res = trained_model.predict(source=str(img_path), conf=0.30, verbose=False)[0]
    plotted = res.plot()
    
    is_defect = len(res.boxes) > 0
    status_text = '❌ REJECT (Defect Detected)' if is_defect else '✅ PASS (Normal)'
    box_color = 'red' if is_defect else 'green'
    
    plt.subplot(2, 2, i + 1)
    plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    plt.title(f"{img_path.name}\nDecision: {status_text}", fontsize=11, color=box_color, weight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()
print("✨ انتهت جميع المراحل بنجاح! يمكنك الآن تنزيل industrial_qc_model_package.zip من قسم Output.")